# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:
# Task 1 — GDP per capita vs life expectancy in Asia (2002 and 2007)
# Two colours represent the two years. Thin grey connectors show each country's direction of change.

continent_choice = "Asia"
years_to_compare = [2002, 2007]

task1_data = df.loc[
    (df["continent"] == continent_choice) &
    (df["year"].isin(years_to_compare))
].copy()

# Keep countries that have observations in both years so every connector is meaningful.
countries_both_years = (
    task1_data.groupby("country")["year"].nunique()
    .loc[lambda counts: counts == len(years_to_compare)]
    .index
)
task1_data = task1_data[task1_data["country"].isin(countries_both_years)].copy()
task1_data["year_label"] = task1_data["year"].astype(str)

if task1_data.empty:
    raise ValueError("No countries have observations for both selected years.")

fig1 = px.scatter(
    task1_data,
    x="gdpPercap",
    y="lifeExp",
    color="year_label",
    color_discrete_map={"2002": "#2878B5", "2007": "#E07A2D"},
    hover_name="country",
    hover_data={
        "year": True,
        "gdpPercap": ":,.0f",
        "lifeExp": ":.1f",
        "pop": ":,",
        "continent": False
    },
    category_orders={"year_label": ["2002", "2007"]},
    labels={
        "gdpPercap": "GDP per capita (international $, log scale)",
        "lifeExp": "Life expectancy (years)",
        "year_label": "Year"
    }
)

# Add light connectors between each country's 2002 and 2007 positions.
for country, country_data in task1_data.groupby("country"):
    country_data = country_data.sort_values("year")
    start = country_data[country_data["year"] == 2002].iloc[0]
    end = country_data[country_data["year"] == 2007].iloc[0]
    fig1.add_trace(
        go.Scatter(
            x=country_data["gdpPercap"],
            y=country_data["lifeExp"],
            mode="lines",
            line=dict(color="#D0D0D0", width=1),
            hoverinfo="skip",
            showlegend=False
        )
    )

# Calculate and annotate the movement in the continent's median values.
median_by_year = (
    task1_data.groupby("year")[["gdpPercap", "lifeExp"]]
    .median()
    .sort_index()
)
median_2002 = median_by_year.loc[2002]
median_2007 = median_by_year.loc[2007]

fig1.add_annotation(
    x=median_2007["gdpPercap"],
    y=median_2007["lifeExp"],
    ax=median_2002["gdpPercap"],
    ay=median_2002["lifeExp"],
    xref="x",
    yref="y",
    axref="x",
    ayref="y",
    text="Median movement, 2002 → 2007",
    showarrow=True,
    arrowhead=3,
    arrowwidth=2,
    arrowcolor="#333333",
    font=dict(size=11, color="#333333"),
    bgcolor="white"
)

fig1.update_xaxes(type="log", showgrid=False, showline=True, linecolor="#CCCCCC")
fig1.update_yaxes(showgrid=True, gridcolor="#F0F0F0", zeroline=False)
fig1.update_layout(
    title=dict(
        text=(
            "Across Asia, countries generally moved toward higher income and longer lives"
            "<br><sup>GDP per capita and life expectancy in 2002 and 2007; connectors show each country's change</sup>"
        ),
        x=0.02,
        xanchor="left"
    ),
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend_title_text="Year",
    margin=dict(l=70, r=35, t=100, b=65)
)
fig1.show()

print(f"Continent: {continent_choice}")
print(f"Countries with data in both years: {task1_data['country'].nunique()}")
print(
    "Median life expectancy:",
    f"{median_2002['lifeExp']:.1f} years (2002) → {median_2007['lifeExp']:.1f} years (2007)"
)
print(
    "Median GDP per capita:",
    f"${median_2002['gdpPercap']:,.0f} (2002) → ${median_2007['gdpPercap']:,.0f} (2007)"
)

Continent: Asia
Countries with data in both years: 33
Median life expectancy: 71.0 years (2002) → 72.4 years (2007)
Median GDP per capita: $4,091 (2002) → $4,471 (2007)


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [4]:
# Task 2 — 2007 bubble chart: Asia highlighted, all other continents grey

bubble_data = df.loc[df["year"] == 2007].copy()
highlight_continent = "Asia"

if bubble_data.empty:
    raise ValueError("The dataset contains no observations for 2007.")

# Draw all non-Asian countries in grey first.
context_data = bubble_data[bubble_data["continent"] != highlight_continent]
highlight_data = bubble_data[bubble_data["continent"] == highlight_continent]

fig2 = px.scatter(
    context_data,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    size_max=28,
    hover_name="country",
    hover_data={
        "gdpPercap": ":,.0f",
        "lifeExp": ":.1f",
        "pop": ":,",
        "continent": True
    },
    labels={
        "gdpPercap": "GDP per capita (international $, log scale)",
        "lifeExp": "Life expectancy (years)",
        "pop": "Population"
    }
)
fig2.update_traces(
    marker=dict(color="#D9D9D9", opacity=0.65, line=dict(width=0.4, color="white")),
    showlegend=False
)

# Add Asia in one accent colour.
fig2.add_trace(
    go.Scatter(
        x=highlight_data["gdpPercap"],
        y=highlight_data["lifeExp"],
        mode="markers",
        name=highlight_continent,
        marker=dict(
            size=highlight_data["pop"],
            sizemode="area",
            sizeref=2.0 * bubble_data["pop"].max() / (28 ** 2),
            sizemin=5,
            color="#D95F02",
            opacity=0.82,
            line=dict(width=0.7, color="white")
        ),
        text=highlight_data["country"],
        customdata=highlight_data[["pop", "continent"]].to_numpy(),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "GDP per capita: $%{x:,.0f}<br>"
            "Life expectancy: %{y:.1f} years<br>"
            "Population: %{customdata[0]:,.0f}<br>"
            "Continent: %{customdata[1]}<extra></extra>"
        )
    )
)

# Use the median of the highlighted group to position a clear explanatory annotation.
asia_median_gdp = highlight_data["gdpPercap"].median()
asia_median_life = highlight_data["lifeExp"].median()
asia_population = highlight_data["pop"].sum()

fig2.add_annotation(
    x=asia_median_gdp,
    y=asia_median_life,
    text=(
        "Asia<br>"
        "Wide range of incomes<br>"
        "and life expectancies"
    ),
    showarrow=True,
    arrowhead=2,
    ax=95,
    ay=-65,
    bgcolor="white",
    bordercolor="#D95F02",
    borderwidth=1,
    font=dict(size=11, color="#333333")
)

fig2.update_xaxes(type="log", showgrid=False, showline=True, linecolor="#CCCCCC")
fig2.update_yaxes(showgrid=True, gridcolor="#F0F0F0", zeroline=False)
fig2.update_layout(
    title=dict(
        text=(
            "Asia spans a wide range of incomes and life expectancies"
            "<br><sup>Every country in 2007; bubble size represents population, Asia is highlighted</sup>"
        ),
        x=0.02,
        xanchor="left"
    ),
    xaxis_title="GDP per capita (international $, log scale)",
    yaxis_title="Life expectancy (years)",
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(l=70, r=40, t=100, b=65)
)
fig2.show()

print(f"Countries shown: {len(bubble_data)}")
print(f"Highlighted continent: {highlight_continent} ({len(highlight_data)} countries)")
print(f"Combined population of highlighted countries: {asia_population:,.0f}")
print(f"Median life expectancy in {highlight_continent}: {asia_median_life:.1f} years")
print(f"Median GDP per capita in {highlight_continent}: ${asia_median_gdp:,.0f}")

Countries shown: 142
Highlighted continent: Asia (33 countries)
Combined population of highlighted countries: 3,811,953,827
Median life expectancy in Asia: 72.4 years
Median GDP per capita in Asia: $4,471


---

## Submission checklist

- Task 1 compares Asian countries in 2002 and 2007, using two year colours and connectors to show direction of change.
- Task 2 uses the full 2007 dataset, log-scaled GDP per capita, population-sized bubbles, and Asia highlighted against grey context.
- Run all cells from top to bottom and save the notebook to `week04/lecture04_exercise.ipynb` in your GitHub repository.
